In [2]:
import sys
import os
import pandas as pd

# Navigate two levels up from machine_learning/notebooks to root URBAN PULSE AI directory
root_dir = os.path.abspath("../..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from machine_learning.preprocessing import (
    load_and_clean_data,
    resample_to_hourly_grid,
    train_test_chronological_split
)
from machine_learning.feature_engineering import build_feature_pipeline
from machine_learning.train import evaluate_naive_persistence, save_baseline_artifacts

In [6]:
# 1. Load raw data using your dataset path
raw_df = load_and_clean_data(file_path="../../data/raw_sample_data.csv")

# 2. Resample to strict hourly grid
hourly_df = resample_to_hourly_grid(raw_df)

# 3. Chronological Train/Test Split (80% train / 20% test)
train_df, test_df = train_test_chronological_split(hourly_df, test_ratio=0.2)

print(f"Train set shape: {train_df.shape}")
print(f"Test set shape:  {test_df.shape}")

Train set shape: (80, 8)
Test set shape:  (20, 8)


In [7]:
# Generate temporal, lag, and rolling features on the dataset
target_pollutants = ['pm10', 'pm25']

engineered_df = build_feature_pipeline(
    df=hourly_df,
    target_cols=target_pollutants,
    lags=[1, 2, 3, 6, 12, 24],
    windows=[3, 6, 12, 24],
    drop_na=True
)

engineered_df.head()

city                        bengaluru                 chennai              \
parameter                        pm10        pm25        pm10        pm25   
timestamp                                                                   
2026-10-01 00:00:00+00:00  127.580582   95.537338  103.600171   59.667981   
2026-10-01 01:00:00+00:00  103.456986  125.166005  137.860173   98.005722   
2026-10-01 02:00:00+00:00   40.343572  113.162324   84.755446  136.343463   
2026-10-01 03:00:00+00:00   93.226687   42.472593   16.123470   78.040069   
2026-10-01 04:00:00+00:00  106.941061   69.947946  106.948996  101.320831   

city                            delhi                  mumbai              \
parameter                        pm10        pm25        pm10        pm25   
timestamp                                                                   
2026-10-01 00:00:00+00:00   93.812188  143.100003   35.423680   45.251868   
2026-10-01 01:00:00+00:00   82.142245  109.130161   80.824415   31.469196   
2026-10-01 02:00:00+00:00   70.472303   52.593914  109.013409  107.531115   
2026-10-01 03:00:00+00:00   37.954330   73.849798   73.117512  140.811382   
2026-10-01 04:00:00+00:00  142.843975   33.873377   92.778561  131.785491   

city                      hour dayofweek month  sin_hour  cos_hour  \
parameter                                                            
timestamp                                                            
2026-10-01 00:00:00+00:00    0         3    10  0.000000  1.000000   
2026-10-01 01:00:00+00:00    1         3    10  0.258819  0.965926   
2026-10-01 02:00:00+00:00    2         3    10  0.500000  0.866025   
2026-10-01 03:00:00+00:00    3         3    10  0.707107  0.707107   
2026-10-01 04:00:00+00:00    4         3    10  0.866025  0.500000   

city                      sin_dayofweek cos_dayofweek sin_month cos_month  
parameter                                                                  
timestamp                                                                  
2026-10-01 00:00:00+00:00      0.433884     -0.900969 -0.866025       0.5  
2026-10-01 01:00:00+00:00      0.433884     -0.900969 -0.866025       0.5  
2026-10-01 02:00:00+00:00      0.433884     -0.900969 -0.866025       0.5  
2026-10-01 03:00:00+00:00      0.433884     -0.900969 -0.866025       0.5  
2026-10-01 04:00:00+00:00      0.433884     -0.900969 -0.866025       0.5

In [8]:
# Compute Naive Persistence baseline metrics on the test partition
baseline_metrics = evaluate_naive_persistence(
    test_df=test_df,
    target_cols=target_pollutants,
    horizons=[1, 24]
)

# Persist metrics artifact to JSON
artifact_path = save_baseline_artifacts(
    metrics=baseline_metrics,
    artifact_dir="../artifacts",
    filename="baseline_metrics.json"
)


✅ Artifact successfully saved to: ../artifacts/baseline_metrics.json
